<a href="https://colab.research.google.com/github/Roua-Elayeb/Analyse-_de_donne-IIOT/blob/main/RouaElayeb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.ensemble import  IsolationForest
import matplotlib.pyplot as plt

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
df = pd.read_csv("compressors_telemetry.csv", parse_dates=["timestamp"],
dtype={"asset_id": "category"})

In [9]:
df.dtypes

,0
asset_id,category
timestamp,datetime64[ns]
vibration_rms_mm_s,float64
temperature_palier_C,float64
pression_aspiration_bar,float64
pression_refoulement_bar,float64
debit_m3_h,float64


**A.1 — Chargement et exploration initiale**

In [32]:

print(df.columns.tolist())

print(df.head())
print(df.dtypes)


['asset_id', 'timestamp', 'vibration_rms_mm_s', 'temperature_palier_C', 'pression_aspiration_bar', 'pression_refoulement_bar', 'debit_m3_h']
  asset_id           timestamp  vibration_rms_mm_s  temperature_palier_C  \
0     C-01 2025-01-01 00:00:00            2.695242             62.858544   
1     C-01 2025-01-01 00:05:00            2.551069             63.140021   
2     C-01 2025-01-01 00:10:00            2.806736             62.957534   
3     C-01 2025-01-01 00:15:00            2.758015             62.425324   
4     C-01 2025-01-01 00:20:00            2.977456             63.035964   

   pression_aspiration_bar  pression_refoulement_bar  debit_m3_h  
0                 0.989603                  7.557338   41.582457  
1                 0.966677                  7.480369   42.215718  
2                 0.958245                  7.615764   43.022345  
3                 1.002675                  7.677110   42.140012  
4                 1.019795                  7.660826   42.799642  


A.1.1

In [6]:
print(f"lignes : {len(df):,}")

lignes : 243,879


In [7]:
print(f"compresseurs : {df['asset_id'].unique()} -> {df ['asset_id'].unique()} ")

compresseurs : ['C-01', 'C-02', 'C-03', 'C-04', 'C-05', 'C-06', 'C-07', 'C-08']
Categories (8, object): ['C-01', 'C-02', 'C-03', 'C-04', 'C-05', 'C-06', 'C-07', 'C-08'] -> ['C-01', 'C-02', 'C-03', 'C-04', 'C-05', 'C-06', 'C-07', 'C-08']
Categories (8, object): ['C-01', 'C-02', 'C-03', 'C-04', 'C-05', 'C-06', 'C-07', 'C-08'] 


In [8]:
nombre_capteur = [col for col in df.columns if col not in ['asset_id', 'timestamp']]
print(f"nombre_capteur : {len(nombre_capteur)}")

nombre_capteur : 5


In [9]:
print(f"periode_min:{df['timestamp'].min()} -> periode_max:{df['timestamp'].max()}")

periode_min:2025-01-01 00:00:00 -> periode_max:2025-04-30 23:55:00


A.1.2

In [10]:
df = df.sort_values(['asset_id', 'timestamp'])
delta = df.groupby('asset_id')['timestamp'].diff().dt.total_seconds()

print("Écart temporel le plus fréquent :", delta.mode().iloc[0], "secondes")
print("Nombre de gaps (> 60 s) :", (delta > 60).sum())

Écart temporel le plus fréquent : 300.0 secondes
Nombre de gaps (> 60 s) : 242142


/tmp/ipykernel_5704/3163601684.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  delta = df.groupby('asset_id')['timestamp'].diff().dt.total_seconds()


ce code calcule le temps entre deux mesures en secondes et détecte les gaps supérieurs à 60 secondes.
donc il y a 276472> 60s

In [11]:
print("Nombre de lignes dupliquées :", df.duplicated().sum())

print("Nombre de timestamps dupliqués par machine :",
      df.duplicated(subset=['asset_id', 'timestamp']).sum())

Nombre de lignes dupliquées : 1728
Nombre de timestamps dupliqués par machine : 1728


On a calculé le nombre de lignes répétées dans le dataset, puis on a vérifié si un même compresseur possède plusieurs mesures avec le même timestamp. On a trouvé 1 728 lignes dupliquées et 1 728 timestamps dupliqués. Cela veut dire qu’il existe des enregistrements répétés qu’il faut vérifier pour comprendre leur origine.

In [12]:
df = df.sort_values(['asset_id', 'timestamp'])
delta = df.groupby('asset_id')['timestamp'].diff()

print("Nombre de timestamps non croissants :", (delta <= pd.Timedelta(0)).sum())

Nombre de timestamps non croissants : 1728


/tmp/ipykernel_5704/4280332140.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  delta = df.groupby('asset_id')['timestamp'].diff()


On a calculé le temps entre deux mesures du même compresseur, puis on a vérifié si les timestamps sont croissants. On a trouvé 1 728 timestamps non croissants. Cela veut dire qu’il existe des mesures avec le même timestamp ou des timestamps qui ne suivent pas l’ordre chronologique. Il faut vérifier ces enregistrements pour comprendre leur origine.

In [13]:
delta_original = df.groupby('asset_id', sort=False)['timestamp'].diff()

print("Nombre de timestamps non croissants :",
      (delta_original <= pd.Timedelta(0)).sum())

Nombre de timestamps non croissants : 1728


/tmp/ipykernel_5704/372576385.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  delta_original = df.groupby('asset_id', sort=False)['timestamp'].diff()


On a calculé le temps entre deux mesures du même compresseur dans l’ordre original du dataset, puis on a vérifié si les timestamps suivent l’ordre chronologique. On a trouvé 1 728 timestamps non croissants. Cela signifie qu’il existe des timestamps identiques ou des mesures qui ne suivent pas l’ordre chronologique. Il faut vérifier ces données pour comprendre leur origine.

A.1.3

In [14]:
SENSORS = [
    'vibration_rms_mm_s',
    'temperature_palier_C',
    'pression_aspiration_bar',
    'pression_refoulement_bar',
    'debit_m3_h'
]

completude = df.groupby('asset_id', observed=False)[SENSORS].apply(
    lambda x: x.notna().mean() * 100
)

print(completude.round(2))

          vibration_rms_mm_s  temperature_palier_C  pression_aspiration_bar  \
asset_id                                                                      
C-01                  100.00                100.00                   100.00   
C-02                  100.00                100.00                   100.00   
C-03                   93.33                100.00                   100.00   
C-04                  100.00                100.00                   100.00   
C-05                  100.00                100.00                   100.00   
C-06                  100.00                100.00                   100.00   
C-07                  100.00                 99.70                   100.00   
C-08                   99.57                 99.57                    99.57   

          pression_refoulement_bar  debit_m3_h  
asset_id                                        
C-01                        100.00      100.00  
C-02                        100.00      100.00  
C-03         

Ce code calcule le taux de complétude de chaque capteur pour chaque compresseur. Il vérifie le pourcentage de valeurs présentes dans chaque colonne. Un taux de 100 % signifie qu’il ne manque aucune valeur, tandis qu’un taux faible indique beaucoup de valeurs manquantes.
le compresseur C-03 présente un taux de complétude de 93,33 % pour le capteur vibration_rms_mm_s, tandis que C-05 présente 97,63 % pour le capteur debit_m3_h

In [15]:
completude_long = completude.reset_index().melt(
    id_vars='asset_id',
    var_name='Capteur',
    value_name='Taux de complétude (%)'
)

print(completude_long.sort_values(
    'Taux de complétude (%)'
).head(5))

   asset_id                   Capteur  Taux de complétude (%)
2      C-03        vibration_rms_mm_s               93.333333
31     C-08  pression_refoulement_bar               96.536797
36     C-05                debit_m3_h               97.632826
15     C-08      temperature_palier_C               99.567100
23     C-08   pression_aspiration_bar               99.567100


On a classé les combinaisons compresseur × capteur selon leur taux de complétude et on a identifié les 5 cas les plus problématiques. Le compresseur C-03 est le plus problématique, avec un taux de complétude de 93,33 % pour les vibrations. Ensuite, on trouve le capteur de pression du compresseur C-08 avec 96,32 %, puis le débit de C-05 avec 97,63 % et la température de C-07 avec 99,70 %. Le dernier cas, C-05 pour les vibrations, affiche 100 % : il n’est donc pas problématique et ne devrait pas apparaître parmi les 5 cas les plus faibles. Cela peut être dû à des ex æquo ou à une erreur dans le tableau affiché.

In [16]:
# Vérifier si la pression d'aspiration est supérieure
# à la pression de refoulement
erreurs_pression = df[
    df['pression_aspiration_bar'] >
    df['pression_refoulement_bar']
]

print("Nombre de valeurs impossibles :", len(erreurs_pression))
display(erreurs_pression)

Nombre de valeurs impossibles : 34560


,asset_id,timestamp,vibration_rms_mm_s,temperature_palier_C,pression_aspiration_bar,pression_refoulement_bar,debit_m3_h
34560,C-02,2025-01-01 00:00:00,2.817836,63.528793,7.698368,1.055544,42.984506
34561,C-02,2025-01-01 00:05:00,2.946242,63.336010,7.660980,1.000195,42.350011
34562,C-02,2025-01-01 00:10:00,2.857264,62.424534,7.738262,0.966711,42.251791
34563,C-02,2025-01-01 00:15:00,2.934514,63.059825,7.640568,1.005997,43.223180
34564,C-02,2025-01-01 00:20:00,2.630248,63.160774,7.500578,1.019304,41.471828
...,...,...,...,...,...,...,...
69115,C-02,2025-04-30 23:35:00,2.760314,62.096733,7.645287,0.994807,42.015499
69116,C-02,2025-04-30 23:40:00,3.203015,62.306812,7.786214,0.987321,42.650010
69117,C-02,2025-04-30 23:45:00,3.129682,61.762819,7.365387,0.974040,41.807526
69118,C-02,2025-04-30 23:50:00,3.186182,61.898427,7.448252,0.967233,42.065425


On a comparé la pression d’aspiration avec la pression de refoulement et on a trouvé 34 560 valeurs anormales. Pour le compresseur C-02, par exemple, la pression d’aspiration est d’environ 7,7 bar, alors que la pression de refoulement est d’environ 1 bar. Cela ne respecte pas la relation physique attendue pour un compresseur en fonctionnement normal. Il faut vérifier les données et les capteurs pour comprendre l’origine de cette anomalie.

In [17]:
print("Pressions négatives :",
      (df[['pression_aspiration_bar',
           'pression_refoulement_bar']] < 0).sum())

print("Températures négatives :",
      (df['temperature_palier_C'] < 0).sum())

print("Vibrations négatives :",
      (df['vibration_rms_mm_s'] < 0).sum())

print("Débits négatifs :",
      (df['debit_m3_h'] < 0).sum())

Pressions négatives : pression_aspiration_bar     0
pression_refoulement_bar    0
dtype: int64
Températures négatives : 0
Vibrations négatives : 0
Débits négatifs : 0


On a vérifié les valeurs négatives des pressions, des températures, des vibrations et des débits. On n’a trouvé aucune valeur négative. Cela signifie qu’il n’y a pas d’anomalies selon ce critère. Cependant, l’anomalie entre la pression d’aspiration et la pression de refoulement reste à vérifier.

**A.1.5**

Non, df.describe() ne suffit pas pour trouver tous les problèmes.

. Exemple 1 : On a trouvé 34 560 cas où la pression d’aspiration est supérieure à la pression de refoulement.

. Exemple 2 : On a trouvé 1 728 lignes répétées dans le dataset.

Donc, il faut faire d’autres vérifications pour trouver tous les problèmes.

**A.2 — Missing values : 4 défauts cachés**

**A.2.1**

In [18]:
SENSORS = [
    'vibration_rms_mm_s',
    'temperature_palier_C',
    'pression_aspiration_bar',
    'pression_refoulement_bar',
    'debit_m3_h'
]

df = df.sort_values(['asset_id', 'timestamp'])

delta = df.groupby('asset_id')['timestamp'].diff()

print("Écarts temporels les plus fréquents :")
print(delta.value_counts().head())

print("\nValeurs manquantes :")
print(df.groupby('asset_id')[SENSORS].apply(lambda x: x.isna().sum()))

Écarts temporels les plus fréquents :
timestamp
0 days 00:05:00    242142
0 days 00:00:00      1728
Name: count, dtype: int64

Valeurs manquantes :
          vibration_rms_mm_s  temperature_palier_C  pression_aspiration_bar  \
asset_id                                                                      
C-01                       0                     0                        0   
C-02                       0                     0                        0   
C-03                    2304                     0                        0   
C-04                       0                     0                        0   
C-05                       0                     0                        0   
C-06                       0                     0                        0   
C-07                       0                   103                        0   
C-08                       1                     1                        1   

          pression_refoulement_bar  debit_m3_h  
asset_id    

/tmp/ipykernel_5704/1334746457.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  delta = df.groupby('asset_id')['timestamp'].diff()
/tmp/ipykernel_5704/1334746457.py:19: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('asset_id')[SENSORS].apply(lambda x: x.isna().sum()))


On a trouvé que les mesures sont normalement prises toutes les 5 minutes, mais il existe aussi des timestamps répétés. Pour les valeurs manquantes, le compresseur C-03 a 2 304 valeurs manquantes dans les vibrations, C-05 a 859 valeurs manquantes dans le débit, et C-07 a 103 valeurs manquantes dans la température. Les autres valeurs manquantes sont peu nombreuses. Il faut vérifier si ces valeurs manquantes sont isolées ou regroupées dans le temps pour mieux comprendre leur nature.

In [19]:
# C-05
c05 = df[df['asset_id'] == 'C-05'].copy()

# les valeurs manquantes du débit
missing = c05[c05['debit_m3_h'].isna()].copy()

# Jour
missing['jour'] = missing['timestamp'].dt.day_name()

# Heure
missing['heure'] = missing['timestamp'].dt.hour

print("Valeurs manquantes par jour :")
print(missing['jour'].value_counts())

print("\nValeurs manquantes par heure :")
print(missing['heure'].value_counts().sort_index())

Valeurs manquantes par jour :
jour
Sunday    859
Name: count, dtype: int64

Valeurs manquantes par heure :
heure
2    221
3    213
4    215
5    210
Name: count, dtype: int64


les 859 valeurs manquantes apparaissent toutes le dimanche, entre 2 h et 5 h du matin. Cela peut être lié à un arrêt programmé ou à une maintenance du capteur pendant cette période

In [25]:
# Définition des variables requises pour l'imputation
sensor_type = "vibration"
max_gap_min = 30  # Seuil de tolérance exemple (ex: 30 minutes)

# Dans notre DataFrame, chaque mesure est espacée de 5 minutes par défaut
# Donc un gap de 30 minutes correspond à un maximum de 6 valeurs manquantes consécutives à interpoler
limit_consecutive = max_gap_min // 5

if sensor_type == "vibration":
    # Appliquer l'interpolation linéaire directement sur la colonne vibration_rms_mm_s
    df["vibration_rms_mm_s"] = df["vibration_rms_mm_s"].interpolate(
        method="linear",
        limit=limit_consecutive
    )
    print("Interpolation appliquée avec succès sur 'vibration_rms_mm_s'.")

Interpolation appliquée avec succès sur 'vibration_rms_mm_s'.


Ce code remplit les valeurs manquantes du capteur de vibration en calculant des valeurs entre celles qui existent déjà. Il le fait seulement pour les petits trous, jusqu’à 30 minutes.

In [26]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Filtrer le compresseur C-03
c03 = df[df["asset_id"] == "C-03"].sort_values("timestamp").copy()

# Identifier les valeurs manquantes de vibration
c03["missing"] = c03["vibration_rms_mm_s"].isna()

# Identifier le début de chaque bloc de valeurs manquantes
c03["bloc"] = (
    c03["missing"] != c03["missing"].shift()
).cumsum()

# Afficher la durée de chaque bloc manquant
blocs = c03[c03["missing"]].groupby("bloc").agg(
    debut=("timestamp", "min"),
    fin=("timestamp", "max"),
    nombre_valeurs=("timestamp", "size")
)

print(blocs)

                   debut                 fin  nombre_valeurs
bloc                                                        
2    2025-01-21 01:00:00 2025-01-22 23:55:00             564
4    2025-02-15 01:00:00 2025-02-16 23:55:00             564
6    2025-03-12 01:00:00 2025-03-13 23:55:00             564
8    2025-04-06 01:00:00 2025-04-07 23:55:00             564


Ce code cherche les moments où les mesures de vibration du compresseur C-03 sont manquantes et calcule combien de temps elles durent. Le résultat montre 4 longues périodes, chacune avec 564 valeurs manquantes, donc elles sont trop longues pour être remplies facilement.

In [28]:
# Filtrer le compresseur C-08
c08 = df[df["asset_id"] == "C-08"].copy()

# Extraire les mesures manquantes de pression (utilisation du bon nom de colonne)
missing = c08[c08["pression_refoulement_bar"].isna()].copy()

# Examiner les dates et heures concernées
missing["jour"] = missing["timestamp"].dt.day_name()
missing["heure"] = missing["timestamp"].dt.hour

print("Nombre de valeurs manquantes :", len(missing))
print("\nRépartition par jour :")
print(missing["jour"].value_counts())

print("\nRépartition par heure :")
print(missing["heure"].value_counts().sort_index())

print("\nDates et heures exactes :")
print(missing["timestamp"].sort_values().to_string(index=False))

Nombre de valeurs manquantes : 8

Répartition par jour :
jour
Wednesday    7
Name: count, dtype: int64

Répartition par heure :
heure
6.0     4
14.0    3
Name: count, dtype: int64

Dates et heures exactes :
2025-01-01 06:15:00
2025-01-01 06:25:00
2025-01-01 06:35:00
2025-01-01 06:40:00
2025-01-01 14:00:00
2025-01-01 14:05:00
2025-01-01 14:15:00
                NaT


Ce code cherche les mesures de pression manquantes du compresseur C-08 et regarde à quels jours et à quelles heures elles arrivent. Il trouve 8 valeurs manquantes, dont 7 le mercredi, et une ligne sans date (NaT).

In [33]:

def impute_iot(df, sensor_type, max_gap_min=5):
    df = df.sort_values("timestamp").copy()

    # Créer was_missing AVANT l'imputation
    df["was_missing"] = df["value"].isna().astype(int)

    if sensor_type == "debit":
        return df

    elif sensor_type == "vibration":
        df["value"] = df["value"].interpolate(
            method="linear",
            limit=max_gap_min,
            limit_area="inside"
        )

    elif sensor_type == "temperature":
        df["value"] = df["value"].ffill(limit=10)
        df["value"] = df["value"].interpolate(
            method="linear",
            limit_area="inside"
        )

    elif sensor_type == "pression":
        df["value"] = df["value"].interpolate(
            method="linear",
            limit=max_gap_min,
            limit_area="inside"
        )

    else:
        raise ValueError("Type de capteur inconnu")

    return df


In [36]:

def impute_iot(df, sensor_type, max_gap_min=5):
    df = df.sort_values("timestamp").copy()

    # Obligatoire : créer was_missing AVANT l'imputation
    df["was_missing"] = df["value"].isna().astype(int)

    if sensor_type == "debit":
        return df

    # Une mesure toutes les 5 minutes
    limit = max(1, max_gap_min // 5)

    if sensor_type == "vibration":
        df["value"] = df["value"].interpolate(
            method="linear",
            limit=limit,
            limit_area="inside"
        )

    elif sensor_type == "temperature":
        df["value"] = df["value"].ffill(limit=10)
        df["value"] = df["value"].interpolate(
            method="linear",
            limit_area="inside"
        )

    elif sensor_type == "pression":
        df["value"] = df["value"].interpolate(
            method="linear",
            limit=limit,
            limit_area="inside"
        )

    else:
        raise ValueError("Type de capteur inconnu")

    return df


# Associer chaque type à sa colonne
capteurs = {
    "vibration": "vibration_rms_mm_s",
    "temperature": "temperature_palier_C",
    "pression_aspiration": "pression_aspiration_bar",
    "pression_refoulement": "pression_refoulement_bar",
    "debit": "debit_m3_h"
}

resultats = []
resume = []

# Traiter chaque compresseur et chaque capteur
for compresseur, groupe in df.groupby("asset_id", observed=True):
    for type_capteur, colonne in capteurs.items():

        donnees = groupe[["timestamp", colonne]].rename(
            columns={colonne: "value"}
        )

        resultat = impute_iot(donnees, type_capteur.split("_")[0])

        resultat["asset_id"] = compresseur
        resultat["capteur"] = type_capteur

        avant = resultat["was_missing"].sum()
        apres = resultat["value"].isna().sum()

        resume.append({
            "compresseur": compresseur,
            "capteur": type_capteur,
            "missing_avant": int(avant),
            "missing_apres": int(apres),
            "valeurs_imputees": int(avant - apres)
        })

        resultats.append(resultat)

# Afficher le bilan
bilan = pd.DataFrame(resume)
display(bilan)

print("Total des valeurs manquantes avant :",
      bilan["missing_avant"].sum())

print("Total des valeurs manquantes après :",
      bilan["missing_apres"].sum())

print("Total des valeurs imputées :",
      bilan["valeurs_imputees"].sum())


,compresseur,capteur,missing_avant,missing_apres,valeurs_imputees
0,C-01,vibration,0,0,0
1,C-01,temperature,0,0,0
2,C-01,pression_aspiration,0,0,0
3,C-01,pression_refoulement,0,0,0
4,C-01,debit,0,0,0
5,C-02,vibration,0,0,0
6,C-02,temperature,0,0,0
7,C-02,pression_aspiration,0,0,0
8,C-02,pression_refoulement,0,0,0
9,C-02,debit,0,0,0


Total des valeurs manquantes avant : 3229
Total des valeurs manquantes après : 3116
Total des valeurs imputées : 113


La fonction impute_iot() a imputé 113 valeurs sur 3 229 valeurs manquantes. Il reste donc 3 116 valeurs manquantes, car les gaps trop longs ne sont pas toujours imputés pour éviter des résultats peu fiables.